# 03 — Feature Selection & Model Training
**Purpose:** Load clean, encoded splits from `data/processed/`, perform statistical
feature selection to identify the most predictive features, train three baseline models
inside sklearn Pipelines, compare their performance, and save the best model.

**Data flow:** `data/processed/` (output of Notebook 02) → feature selection
→ sklearn Pipeline → model evaluation → `models/best_model_v1.joblib`

**What you will learn:**
- How to use Pearson correlation, ANOVA F-test, and Mutual Information to rank features
- How to combine multiple statistical tests into a unified ranking
- How to build sklearn `Pipeline` objects for clean, reproducible model training
- How to diagnose overfitting with train vs. validation metrics
- How to run 5-fold cross-validation for reliable model comparison

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from pathlib import Path
from scipy import stats

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_selection import f_regression, mutual_info_regression
from sklearn.model_selection import cross_val_score
from sklearn import set_config

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.4f}".format)

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR    = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"Processed data dir : {PROCESSED_DIR}")
print(f"Models dir         : {MODELS_DIR}")

📝 **What's happening here**

We import every library needed for the entire notebook in one place so the
dependency list is transparent and easy to audit. Key additions vs. Notebook 02:

| Import | Purpose |
|--------|---------|
| `scipy.stats` | Spearman correlation and ANOVA F-test |
| `f_regression` | sklearn's vectorised Pearson F-test |
| `mutual_info_regression` | Non-linear feature-target dependence |
| `cross_val_score` | 5-fold CV to get a stable RMSE estimate |
| `joblib` | Serialise the final trained pipeline |

`PROCESSED_DIR` points to the clean splits produced by Notebook 02.
`MODELS_DIR` is where the final pipeline will be saved (`best_model_v1.joblib`).

## 1) Load Processed Data

In [ ]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train.csv", index_col=0)
X_val   = pd.read_csv(PROCESSED_DIR / "X_val.csv",   index_col=0)
X_test  = pd.read_csv(PROCESSED_DIR / "X_test.csv",  index_col=0)

y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv", index_col=0).squeeze()
y_val   = pd.read_csv(PROCESSED_DIR / "y_val.csv",   index_col=0).squeeze()
y_test  = pd.read_csv(PROCESSED_DIR / "y_test.csv",  index_col=0).squeeze()

with open(PROCESSED_DIR / "feature_metadata.json") as f:
    meta = json.load(f)

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")
print()
print(f"y_train : {y_train.shape}  mean=${y_train.mean():,.0f}")
print(f"y_val   : {y_val.shape}  mean=${y_val.mean():,.0f}")
print(f"y_test  : {y_test.shape}  mean=${y_test.mean():,.0f}")
print()
print(f"Feature metadata keys: {list(meta.keys())}")

📝 **What's happening here**

We load the six split files produced by Notebook 02 — three feature matrices
(`X_*`) and three target vectors (`y_*`). The `.squeeze()` call converts the
single-column DataFrames from the CSV into plain pandas Series, which is what
sklearn expects for targets.

Key things to check in the output:
- Shapes are consistent: train ~1757 rows, val/test ~586 rows each.
- Mean SalePrice is similar across all three splits (~$180k). Large differences
  would indicate a poorly stratified split.
- `X_train` has many columns because ordinal features are integer-encoded and
  nominal features are OHE-expanded. All columns are numeric.

`feature_metadata.json` from NB02 stores the original column classifications
(ordinal_cols, nominal_cols, engineered features) which we will use to guide
the feature selection analysis.

In [ ]:
# Verify no nulls survived and all columns are numeric
assert X_train.isnull().sum().sum() == 0, "Nulls found in X_train!"
assert X_val.isnull().sum().sum()   == 0, "Nulls found in X_val!"
assert X_test.isnull().sum().sum()  == 0, "Nulls found in X_test!"
assert not X_train.select_dtypes(include="object").columns.tolist(), "Object columns found!"

print("Data quality checks passed:")
print(f"  Zero nulls in all splits ✓")
print(f"  All columns numeric      ✓")
print(f"  Total features: {X_train.shape[1]}")
print()
print("Feature dtype breakdown:")
print(X_train.dtypes.value_counts().to_string())

📝 **What's happening here**

These assertions act as a contract between Notebook 02 (which produced the data)
and Notebook 03 (which consumes it). If Notebook 02 changes and accidentally
leaves nulls or object columns, this cell fails loudly with a clear error message
rather than silently producing wrong model results downstream.

The dtype breakdown shows the mix of float64 (scaled numeric features,
OHE binary columns) and int64 (ordinal-encoded quality features). A purely
numeric matrix means we can pass it directly to any sklearn estimator without
additional preprocessing.

## 2) Statistical Feature Selection

### 2a) Pearson Correlation (Numeric Features)

In [ ]:
# Pearson r between each feature and SalePrice (on training set only)
pearson_r = X_train.corrwith(y_train)
pearson_abs = pearson_r.abs().sort_values(ascending=False)

top30_pearson = pearson_abs.head(30)

fig, ax = plt.subplots(figsize=(10, 7))
top30_pearson.plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(0.5, color="red", linestyle="--", linewidth=1.2, label="|r| = 0.5 threshold")
ax.invert_yaxis()
ax.set_xlabel("|Pearson r|")
ax.set_title("Top 30 Features by |Pearson Correlation| with SalePrice")
ax.legend()
plt.tight_layout()
plt.show()

print("Top 15 features by |Pearson r|:")
print(pearson_abs.head(15).to_string())

📝 **What's happening here**

**Pearson r** measures the strength and direction of the *linear* relationship
between each feature and `SalePrice`. A value of |r| = 1.0 means perfect linear
correlation; |r| = 0 means no linear relationship.

**We compute on training data only** — including val/test would leak target
information into the selection step (target leakage), causing the model to
appear better than it actually is on new data.

**Interpretation:**
- |r| > 0.5 → strong linear predictor — likely very useful
- 0.3 < |r| < 0.5 → moderate correlation — worth considering
- |r| < 0.3 → weak or nonlinear relationship — may not help a linear model

**Limitation:** Pearson only captures *linear* relationships. A feature can have
a strong nonlinear effect on SalePrice and still show a low Pearson r. This is
why we also run Mutual Information (§2c).

### 2b) ANOVA F-Test

In [ ]:
# F-test (sklearn's f_regression) — robust to heteroscedasticity
f_scores, p_values = f_regression(X_train, y_train)

f_df = pd.DataFrame({
    "feature"  : X_train.columns,
    "f_score"  : f_scores,
    "p_value"  : p_values,
}).sort_values("f_score", ascending=False).reset_index(drop=True)

top30_f = f_df.head(30).set_index("feature")

fig, ax = plt.subplots(figsize=(10, 7))
top30_f["f_score"].plot(kind="barh", ax=ax, color="coral")
ax.invert_yaxis()
ax.set_xlabel("F-Score")
ax.set_title("Top 30 Features by ANOVA F-Score")
plt.tight_layout()
plt.show()

print("Top 15 features by F-score:")
print(f_df[["feature", "f_score", "p_value"]].head(15).to_string(index=False))

📝 **What's happening here**

**sklearn's `f_regression`** computes the F-statistic for each feature by fitting
a univariate linear model and testing whether the coefficient is significantly
different from zero. It is equivalent to the squared Pearson correlation scaled
by the degrees of freedom.

**Why use F-test alongside Pearson r?**
The F-score incorporates sample size — a weak correlation in a large dataset can
be highly significant, while a strong correlation in a tiny dataset may not be.
P-values < 0.05 indicate that a feature's relationship with the target is unlikely
to be due to chance.

**Reading the output:**
- High F-score + low p-value → statistically significant linear predictor
- The ranking should closely mirror the Pearson correlation ranking (they both
  capture linear relationships), but F-scores give a better sense of statistical
  confidence.

### 2c) Mutual Information

In [ ]:
# Mutual Information — captures non-linear relationships
mi_scores = mutual_info_regression(X_train, y_train, random_state=42)

mi_df = pd.DataFrame({
    "feature"  : X_train.columns,
    "mi_score" : mi_scores,
}).sort_values("mi_score", ascending=False).reset_index(drop=True)

top30_mi = mi_df.head(30).set_index("feature")

fig, ax = plt.subplots(figsize=(10, 7))
top30_mi["mi_score"].plot(kind="barh", ax=ax, color="mediumseagreen")
ax.invert_yaxis()
ax.set_xlabel("Mutual Information Score")
ax.set_title("Top 30 Features by Mutual Information with SalePrice")
plt.tight_layout()
plt.show()

print("Top 15 features by Mutual Information:")
print(mi_df.head(15).to_string(index=False))

📝 **What's happening here**

**Mutual Information (MI)** measures how much knowing a feature reduces
uncertainty about `SalePrice`. Unlike Pearson r, MI captures *any* statistical
dependency — linear *and* non-linear. A feature that has a threshold effect
(e.g., "houses built after 1980 are worth much more, but within each decade
the effect is flat") will show a low Pearson r but a high MI score.

**MI = 0** means the feature is statistically independent of SalePrice.
**Higher MI** means stronger dependency (no upper bound — unlike r ∈ [-1, 1]).

**What to look for:**
- Features that rank much higher on MI than on Pearson r have *nonlinear*
  relationships with price — valuable for tree-based models (RF, GBR) but not
  for linear regression.
- Features that rank equally high on both tests have strong *linear* relationships
  and are universally useful.

### 2d) Combined Ranking

In [ ]:
# Merge all three tests into a unified ranking
pearson_rank = pearson_abs.rank(ascending=False).rename("pearson_rank")
f_rank       = f_df.set_index("feature")["f_score"].rank(ascending=False).rename("f_rank")
mi_rank      = mi_df.set_index("feature")["mi_score"].rank(ascending=False).rename("mi_rank")

combined = pd.concat([pearson_rank, f_rank, mi_rank], axis=1)
combined["avg_rank"] = combined.mean(axis=1)
combined["pearson_r"] = pearson_r.abs()
combined = combined.sort_values("avg_rank").reset_index()
combined.columns = ["feature", "pearson_rank", "f_rank", "mi_rank", "avg_rank", "pearson_r"]

print("Unified feature ranking (lower avg_rank = stronger predictor):")
print(combined.head(20).to_string(index=False))

📝 **What's happening here**

No single statistical test is definitive. We combine the three tests by:
1. Converting each test's scores to **ranks** (rank 1 = best predictor)
2. Averaging the three ranks into `avg_rank`
3. Sorting by `avg_rank` ascending (lower = more consistently important)

**Why average ranks instead of scores?**
The three tests produce scores on different scales (r ∈ [0, 1], F-score ∈ [0, ∞),
MI ∈ [0, ∞)). Ranking each test separately and then averaging makes the scales
commensurable without requiring normalisation.

**A feature that ranks highly on all three tests is a robust predictor** —
it shows a strong linear relationship (Pearson/F-test), a strong overall
dependency (MI), and is unlikely to be an artefact of any single test's
assumptions.

### 2e) Final Feature Selection

In [ ]:
# Select top 15 features by combined ranking
TOP_N = 15
selected_features = combined["feature"].head(TOP_N).tolist()

print(f"Selected top {TOP_N} features:")
for i, f in enumerate(selected_features, 1):
    row = combined[combined["feature"] == f].iloc[0]
    print(f"  {i:2d}. {f:<35s}  avg_rank={row['avg_rank']:.1f}  |r|={row['pearson_r']:.3f}")

print(f"\nReduced from {X_train.shape[1]} → {len(selected_features)} features")

📝 **What's happening here**

We pick the top 15 features by combined rank. This reduction serves several purposes:

1. **Simpler models** — fewer features mean less overfitting risk, especially for
   linear regression which can be sensitive to multicollinearity.
2. **Faster training** — particularly for cross-validation which fits the model
   multiple times.
3. **Interpretability** — a 15-feature model is much easier to explain to
   stakeholders than a 84-feature model.

**Why 15?** The "top 10% of features" rule of thumb applied to our ~84 features.
In practice, you would sweep this threshold (e.g., 10, 15, 20, 25 features) and
pick the value that maximises val RMSE while keeping the model simple.

The selected features include the key drivers identified across all three tests:
total square footage, overall quality, year built, and garage/basement areas.

In [ ]:
# Subset all splits to selected features
X_tr  = X_train[selected_features].copy()
X_v   = X_val[selected_features].copy()
X_te  = X_test[selected_features].copy()

print("Subsetted split shapes:")
print(f"  X_tr : {X_tr.shape}")
print(f"  X_v  : {X_v.shape}")
print(f"  X_te : {X_te.shape}")
print()
print("Selected feature dtypes:")
print(X_tr.dtypes.value_counts().to_string())

📝 **What's happening here**

We apply the same feature selection to all three splits using the list derived
from the training set only — we never look at val or test to make selection
decisions. Subsetting with a list of column names is safe and will raise a
`KeyError` if any selected feature name is wrong, which is a useful safety check.

## 3) Build sklearn Pipelines

In [ ]:
# All features are already numeric, imputed, and scaled from NB02.
# The Pipeline contains only the model estimator.
# We define one pipeline per model for clean encapsulation.

set_config(display="diagram")

lr_pipeline = Pipeline([("model", LinearRegression())])
rf_pipeline = Pipeline([
    ("model", RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ))
])
gb_pipeline = Pipeline([
    ("model", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42,
        n_iter_no_change=20,
        validation_fraction=0.1,
    ))
])

print("Pipelines defined:")
for name, pl in [("LinearRegression", lr_pipeline),
                 ("RandomForest",     rf_pipeline),
                 ("GradientBoosting", gb_pipeline)]:
    steps = [s[0] for s in pl.steps]
    print(f"  {name:<20s}: steps={steps}")

📝 **What's happening here**

Because Notebook 02 already handled all imputation, encoding, and scaling,
each pipeline here contains a single step: the model estimator. This is
intentional — preprocessing is done once in NB02, and NB03 focuses purely on
model comparison.

**Pipeline hyperparameter choices:**

| Model | Key settings | Rationale |
|-------|-------------|-----------|
| LinearRegression | defaults | Maximum simplicity baseline |
| RandomForestRegressor | `n_estimators=300`, `min_samples_leaf=3` | 300 trees for stable estimates; `min_samples_leaf=3` reduces overfitting |
| GradientBoostingRegressor | `n_estimators=300`, `lr=0.05`, `max_depth=4`, `n_iter_no_change=20` | Slow learning rate + early stopping controls overfitting |

**`n_iter_no_change=20`** enables early stopping: if the model's internal
validation loss does not improve for 20 consecutive rounds, training stops. This
prevents the gradient boosting model from overfitting when more trees stop helping.

In [ ]:
# Display sklearn Pipeline diagram for each model
print("Linear Regression Pipeline:")
display(lr_pipeline)
print("\nRandom Forest Pipeline:")
display(rf_pipeline)
print("\nGradient Boosting Pipeline:")
display(gb_pipeline)

📝 **What's happening here**

`set_config(display="diagram")` activates sklearn's HTML pipeline visualiser.
In a Jupyter notebook, this renders a clickable interactive diagram showing each
pipeline step and its hyperparameters — a quick visual check that the pipeline
is assembled correctly before training begins.

## 4) Evaluation Helper

In [ ]:
def eval_pipeline(name, pipeline, X_tr, y_tr, X_ev, y_ev):
    """Fit a pipeline and return train + eval metrics in a dict.

    Args:
        name:     Model name label for the output dict.
        pipeline: Unfitted sklearn Pipeline (will be fitted inside).
        X_tr:     Training features.
        y_tr:     Training target.
        X_ev:     Evaluation (val or test) features.
        y_ev:     Evaluation target.

    Returns:
        Dict with keys: model, train_mae, train_rmse, train_r2,
        val_mae, val_rmse, val_r2.
    """
    pipeline.fit(X_tr, y_tr)
    train_pred = pipeline.predict(X_tr)
    eval_pred  = pipeline.predict(X_ev)
    return {
        "model":      name,
        "train_mae":  mean_absolute_error(y_tr, train_pred),
        "train_rmse": np.sqrt(mean_squared_error(y_tr, train_pred)),
        "train_r2":   r2_score(y_tr, train_pred),
        "val_mae":    mean_absolute_error(y_ev, eval_pred),
        "val_rmse":   np.sqrt(mean_squared_error(y_ev, eval_pred)),
        "val_r2":     r2_score(y_ev, eval_pred),
    }

print("eval_pipeline() defined.")
print()
print("Output keys and interpretation:")
print("  train_mae   — MAE on training set (optimistic)")
print("  train_rmse  — RMSE on training set (optimistic)")
print("  train_r2    — R² on training set")
print("  val_mae     — MAE on validation set (unbiased)")
print("  val_rmse    — RMSE on validation set ← PRIMARY selection metric")
print("  val_r2      — R² on validation set")
print()
print("Diagnostics:")
print("  train_rmse << val_rmse  → overfitting (model memorised training data)")
print("  both high               → underfitting (model too simple)")
print("  close and low           → good generalisation")

📝 **What's happening here**

We define a reusable helper that encapsulates the fit → predict → score pattern.
Instead of writing the same 10 lines three times (once per model), we call
`eval_pipeline()` three times and collect results into a list.

**Why we track both train and val metrics:**
- *Val metrics alone* cannot tell you whether a model is overfitting. If val
  RMSE = $36k, that could be because the model has high bias (underfitting) or
  because it is overfitting training noise that does not generalise.
- *Train RMSE vs val RMSE gap* is the diagnostic: a large gap means the model
  is memorising training patterns that do not generalise; a small gap with high
  RMSE means the model lacks capacity.

**Why `val_rmse` is the primary selection metric:**
RMSE penalises large errors more heavily than MAE (it squares residuals before
averaging). For housing predictions, a $150k error is genuinely far worse than
three $50k errors — RMSE reflects that asymmetry. MAE treats them as equal.

## 5) Baseline 1 — Linear Regression

In [ ]:
lr_results = eval_pipeline("LinearRegression", lr_pipeline, X_tr, y_train, X_v, y_val)

print(f"Linear Regression Results:")
print(f"  Train  MAE : ${lr_results['train_mae']:>10,.0f}")
print(f"  Train  RMSE: ${lr_results['train_rmse']:>10,.0f}")
print(f"  Train  R²  : {lr_results['train_r2']:>10.4f}")
print(f"  Val    MAE : ${lr_results['val_mae']:>10,.0f}")
print(f"  Val    RMSE: ${lr_results['val_rmse']:>10,.0f}")
print(f"  Val    R²  : {lr_results['val_r2']:>10.4f}")
print()

# 5-fold cross-validation on training set
lr_cv = cross_val_score(lr_pipeline, X_tr, y_train,
                        cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
lr_cv_rmse = -lr_cv
print(f"  5-fold CV RMSE: ${lr_cv_rmse.mean():,.0f} ± ${lr_cv_rmse.std():,.0f}")
print(f"  Folds: {['${:,.0f}'.format(v) for v in lr_cv_rmse]}")

📝 **What's happening here**

**Linear Regression** is always the starting baseline. Any more complex model
that cannot beat this baseline is not worth the added complexity.

**Reading the train-val gap:**
- If train RMSE ≈ val RMSE → the model generalises well; any remaining error
  is irreducible noise or model capacity limits.
- If train RMSE << val RMSE → overfitting.
- If both train RMSE and val RMSE are high → underfitting.

**5-fold cross-validation:** We run CV on the *training set* to get a more
reliable RMSE estimate. A single val split (20% of data = ~586 houses) can
vary substantially by chance. CV averages over 5 different held-out folds,
reducing this variance. The CV RMSE is the number we will use for final model
selection in §9.

## 6) Baseline 2 — Random Forest

In [ ]:
rf_results = eval_pipeline("RandomForest", rf_pipeline, X_tr, y_train, X_v, y_val)

print(f"Random Forest Results:")
print(f"  Train  MAE : ${rf_results['train_mae']:>10,.0f}")
print(f"  Train  RMSE: ${rf_results['train_rmse']:>10,.0f}")
print(f"  Train  R²  : {rf_results['train_r2']:>10.4f}")
print(f"  Val    MAE : ${rf_results['val_mae']:>10,.0f}")
print(f"  Val    RMSE: ${rf_results['val_rmse']:>10,.0f}")
print(f"  Val    R²  : {rf_results['val_r2']:>10.4f}")
print()

rf_cv = cross_val_score(rf_pipeline, X_tr, y_train,
                        cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
rf_cv_rmse = -rf_cv
print(f"  5-fold CV RMSE: ${rf_cv_rmse.mean():,.0f} ± ${rf_cv_rmse.std():,.0f}")
print(f"  Folds: {['${:,.0f}'.format(v) for v in rf_cv_rmse]}")

📝 **What's happening here**

**Random Forest** builds 300 decision trees on random subsets of data and
features, then averages their predictions. This ensemble approach:
- **Reduces variance** compared to a single decision tree
- **Captures nonlinear relationships** that linear regression cannot
- Is naturally robust to outliers (trees split on thresholds, not distances)

**Expected pattern vs. Linear Regression:**
- Train RMSE will be lower (RF has more capacity, fits training data better)
- Val RMSE will likely be lower too (captures real nonlinear signals)
- The train-val gap will be *wider* — more capacity means more potential for
  memorising training noise

The `min_samples_leaf=3` setting prevents the trees from splitting into
tiny leaf nodes (which memorise individual training examples), acting as
a regulariser.

## 7) Baseline 3 — Gradient Boosting

In [ ]:
gb_results = eval_pipeline("GradientBoosting", gb_pipeline, X_tr, y_train, X_v, y_val)

print(f"Gradient Boosting Results:")
print(f"  Train  MAE : ${gb_results['train_mae']:>10,.0f}")
print(f"  Train  RMSE: ${gb_results['train_rmse']:>10,.0f}")
print(f"  Train  R²  : {gb_results['train_r2']:>10.4f}")
print(f"  Val    MAE : ${gb_results['val_mae']:>10,.0f}")
print(f"  Val    RMSE: ${gb_results['val_rmse']:>10,.0f}")
print(f"  Val    R²  : {gb_results['val_r2']:>10.4f}")
print()

gb_cv = cross_val_score(gb_pipeline, X_tr, y_train,
                        cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1)
gb_cv_rmse = -gb_cv
print(f"  5-fold CV RMSE: ${gb_cv_rmse.mean():,.0f} ± ${gb_cv_rmse.std():,.0f}")
print(f"  Folds: {['${:,.0f}'.format(v) for v in gb_cv_rmse]}")

📝 **What's happening here**

**Gradient Boosting** builds trees sequentially, where each new tree tries to
correct the errors of all previous trees. This is more powerful but also more
prone to overfitting than Random Forest:

- **Learning rate = 0.05:** Each tree contributes only 5% of its correction,
  forcing the ensemble to learn slowly and steadily. Lower learning rate + more
  trees = better generalisation but slower training.
- **max_depth = 4:** Shallow trees (max 4 splits) prevent the model from
  memorising rare training patterns.
- **n_iter_no_change = 20:** Early stopping — training halts automatically
  when improvement stagnates, further preventing overfitting.

**Typical pattern:** GB achieves the lowest train RMSE (it can fit the training
data very precisely), a wider train-val gap than RF, but often the best or
near-best val RMSE because it captures complex patterns RF misses.

In [ ]:
# Overfitting diagnostic table: all 3 models
all_results = [lr_results, rf_results, gb_results]

print(f"{'Model':<22} {'Train RMSE':>11} {'Val RMSE':>10} {'Gap':>9} {'Train R²':>9} {'Val R²':>7}")
print("-" * 73)
for r in all_results:
    gap = r["val_rmse"] - r["train_rmse"]
    print(f"{r['model']:<22} {r['train_rmse']:>11,.0f} {r['val_rmse']:>10,.0f} "
          f"{gap:>+9,.0f} {r['train_r2']:>9.3f} {r['val_r2']:>7.3f}")
print()
print("Gap interpretation:")
print("  Large positive gap  → overfitting (model memorised training noise)")
print("  Gap near zero, high RMSE → underfitting (model too simple)")
print("  Gap near zero, low RMSE  → good generalisation")

📝 **What's happening here**

The **overfitting diagnostic table** is one of the most important outputs in
this notebook. Reading across all three rows simultaneously reveals the
**bias-variance tradeoff** in concrete numbers:

- As we move from LinearRegression → RandomForest → GradientBoosting:
  - **Train RMSE decreases**: each model has more capacity and fits training
    data more closely.
  - **The train-val gap grows**: more capacity means more potential for
    memorisation.
  - **Val RMSE decreases**: the models are capturing real signal, not just noise.

This shows that increased model complexity *does* help (val RMSE improves),
but the improvement comes with a cost (larger train-val gap = more overfitting).
Regularisation (Week 3) aims to keep the capacity benefits while reducing the gap.

## 8) Model Comparison

In [ ]:
# Comparison table sorted by val_rmse
comparison = pd.DataFrame(all_results).sort_values("val_rmse")
comparison_display = comparison[["model", "train_rmse", "val_rmse", "train_r2", "val_r2"]].copy()
comparison_display.columns = ["Model", "Train RMSE", "Val RMSE", "Train R²", "Val R²"]
print(comparison_display.to_string(index=False))

📝 **What's happening here**

The comparison table, sorted by `val_rmse` ascending, shows the winner at the
top. We sort by val RMSE (not train RMSE) because val performance is the only
honest measure of how the model will perform on new, unseen houses.

**Model selection rule:** The model with the lowest val RMSE wins, all else
being equal. If two models are within ~$500 RMSE of each other (noise level for
this dataset size), prefer the simpler model (Random Forest over Gradient
Boosting, Linear Regression over Random Forest).

In [ ]:
# Bar charts: Train vs Val RMSE, and Val R²/MAE
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
models = comparison["model"]
x = range(len(models))

# Train vs Val RMSE (side-by-side bars)
axes[0].bar([i - 0.2 for i in x], comparison["train_rmse"],
            width=0.4, label="Train RMSE", color="steelblue")
axes[0].bar([i + 0.2 for i in x], comparison["val_rmse"],
            width=0.4, label="Val RMSE", color="coral")
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15, ha="right")
axes[0].set_ylabel("RMSE ($)")
axes[0].set_title("Train vs Validation RMSE")
axes[0].legend()

# Val R²
axes[1].bar(x, comparison["val_r2"], color=["steelblue", "coral", "mediumseagreen"])
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=15, ha="right")
axes[1].set_ylabel("R²")
axes[1].set_ylim(0, 1)
axes[1].set_title("Validation R²")

# Val MAE
axes[2].bar(x, comparison["val_mae"], color=["steelblue", "coral", "mediumseagreen"])
axes[2].set_xticks(x)
axes[2].set_xticklabels(models, rotation=15, ha="right")
axes[2].set_ylabel("MAE ($)")
axes[2].set_title("Validation MAE")

plt.suptitle("Three-Model Comparison — Validation Metrics", fontsize=13)
plt.tight_layout()
plt.show()

📝 **What's happening here**

Three charts give a complete picture of model performance:

1. **Train vs Val RMSE (side-by-side):** The visual gap between the blue and
   coral bars is the overfitting diagnostic from the table, made spatial.
   A large gap (tall coral bar, short blue bar) is clear evidence of overfitting.

2. **Validation R²:** How much of the price variance does each model explain?
   Higher is better. R² = 1.0 means perfect prediction.

3. **Validation MAE:** The average absolute dollar error on unseen houses.
   MAE is easier to communicate to non-technical stakeholders: "On average,
   our model's predictions are off by $X." 

In [ ]:
# 5-fold CV RMSE comparison — the most reliable model selection metric
cv_results = {
    "LinearRegression": (lr_cv_rmse.mean(), lr_cv_rmse.std()),
    "RandomForest":     (rf_cv_rmse.mean(), rf_cv_rmse.std()),
    "GradientBoosting": (gb_cv_rmse.mean(), gb_cv_rmse.std()),
}

print("5-Fold CV RMSE (training set, lower is better):")
print(f"{'Model':<22} {'CV RMSE':>12} {'± Std':>10}")
print("-" * 46)
for model, (mean, std) in cv_results.items():
    print(f"{model:<22} ${mean:>10,.0f} ± ${std:>8,.0f}")

# Plot CV RMSE with error bars
fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(cv_results))
means = [v[0] for v in cv_results.values()]
stds  = [v[1] for v in cv_results.values()]
bars = ax.bar(x, means, yerr=stds, capsize=5,
              color=["steelblue", "coral", "mediumseagreen"],
              edgecolor="white", linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(list(cv_results.keys()), rotation=10, ha="right")
ax.set_ylabel("CV RMSE ($)")
ax.set_title("5-Fold Cross-Validation RMSE (Primary Selection Metric)")
plt.tight_layout()
plt.show()

# Identify best model by CV RMSE
best_model_name = min(cv_results, key=lambda k: cv_results[k][0])
print(f"\nBest model by CV RMSE: {best_model_name}  (CV RMSE = ${cv_results[best_model_name][0]:,.0f})")

📝 **What's happening here**

**CV RMSE is the primary model selection metric** because it is the most reliable.

**Why CV over a single val split?**
A single val split uses ~586 houses. Which 586 houses ended up in val is random.
A lucky split (many easy-to-predict houses in val) gives optimistically low RMSE;
an unlucky split gives pessimistically high RMSE.

5-fold CV trains and evaluates 5 times, using different held-out folds each time,
then averages. This averages out the luck factor and gives a much more stable
estimate of how the model will perform on genuinely new data.

**Reading error bars:** The ± standard deviation across 5 folds shows how
consistent the model is. A large error bar means the model performs very
differently depending on which houses it is evaluated on — a sign of instability
that should be investigated before deployment.

## 9) Feature Importance (Best Model)

In [ ]:
# Use the best model from CV RMSE comparison
best_pipelines = {
    "LinearRegression": lr_pipeline,
    "RandomForest":     rf_pipeline,
    "GradientBoosting": gb_pipeline,
}
best_pipeline = best_pipelines[best_model_name]

# Extract feature importances (for tree-based models)
best_estimator = best_pipeline.named_steps["model"]

if hasattr(best_estimator, "feature_importances_"):
    importances = pd.Series(
        best_estimator.feature_importances_, index=selected_features
    ).sort_values(ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Top 15
    importances.head(15).plot(kind="barh", ax=axes[0], color="steelblue")
    axes[0].invert_yaxis()
    axes[0].set_title(f"Top 15 Feature Importances ({best_model_name})")
    axes[0].set_xlabel("Importance")

    # Bottom 10 (potential noise)
    importances.tail(min(10, len(importances))).plot(kind="barh", ax=axes[1], color="lightcoral")
    axes[1].invert_yaxis()
    axes[1].set_title("Bottom Features (Lowest Importance)")
    axes[1].set_xlabel("Importance")

    plt.tight_layout()
    plt.show()

    print("Top 10 features by importance:")
    print(importances.head(10).to_string())
    print()
    print("Candidates for removal (bottom 5):")
    print(importances.tail(5).to_string())
else:
    # Linear Regression: use absolute coefficients as importance proxy
    coefs = pd.Series(
        np.abs(best_estimator.coef_), index=selected_features
    ).sort_values(ascending=False)
    coefs.plot(kind="barh", figsize=(10, 6), color="steelblue")
    plt.gca().invert_yaxis()
    plt.title(f"|Coefficient| — {best_model_name}")
    plt.xlabel("|Coefficient| (proxy for importance)")
    plt.tight_layout()
    plt.show()
    print("Top features by |coefficient|:")
    print(coefs.head(10).to_string())

📝 **What's happening here**

**Feature importance** tells us how much each feature contributed to the model's
predictions. For tree-based models (RF, GBR), importance is the total reduction
in mean squared error across all splits where that feature was used.

**Cross-checking with our statistical ranking (§2):**
If the top features here match the top features from Pearson/MI, the model is
learning sensible signals. If the rankings diverge significantly — e.g., a
feature ranks 20th in Pearson but 1st in tree importance — the tree discovered
a nonlinear relationship the linear tests missed.

**Bottom features:** Features with near-zero importance are candidates for
removal. Removing them typically:
- Does not hurt validation RMSE (the model was ignoring them anyway)
- Can *improve* it slightly by reducing noise in the feature matrix
- Speeds up training and inference

In [ ]:
# Residual plot on validation set
val_pred = best_pipeline.predict(X_v)
residuals_val = y_val - val_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Residual vs Predicted
axes[0].scatter(val_pred, residuals_val, alpha=0.4, s=15, color="steelblue")
axes[0].axhline(0, color="red", linewidth=1.5, linestyle="--")
axes[0].set_xlabel("Predicted Sale Price ($)")
axes[0].set_ylabel("Residual (Actual − Predicted)")
axes[0].set_title(f"Residuals vs Predicted — {best_model_name} (Val)")

# Residual distribution
axes[1].hist(residuals_val, bins=40, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", linewidth=1.5, linestyle="--")
axes[1].axvline(residuals_val.mean(), color="orange", linewidth=1.5, linestyle="--",
                label=f"Mean: ${residuals_val.mean():+,.0f}")
axes[1].set_xlabel("Residual ($)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution (Validation Set)")
axes[1].legend()

plt.suptitle(f"Residual Analysis — {best_model_name}", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Residual summary:")
print(f"  Mean  : ${residuals_val.mean():>10,.0f}  (close to 0 = no systematic bias)")
print(f"  Std   : ${residuals_val.std():>10,.0f}")
print(f"  Min   : ${residuals_val.min():>10,.0f}")
print(f"  Max   : ${residuals_val.max():>10,.0f}")

📝 **What's happening here**

Residual plots are the most important diagnostic chart for a regression model.

**Left chart — Residuals vs Predicted:**
- An ideal model has residuals scattered randomly around y = 0 across all
  predicted values.
- A **fan shape** (residuals spreading wider at higher predictions) indicates
  *heteroscedasticity* — the model makes larger errors on expensive houses. This
  is common when predicting a right-skewed target like SalePrice in raw scale.
  A log transform of the target often fixes this.
- A **curve or trend** would indicate a systematic nonlinear pattern the model
  is not capturing.

**Right chart — Residual distribution:**
- Should be roughly bell-shaped and centred near 0.
- A right tail means the model is underestimating some expensive houses.
- A non-zero mean indicates systematic bias (consistently over or under-predicting).

## 10) Final Test Evaluation (Once Only)

In [ ]:
# Evaluate on test set EXACTLY ONCE — the official baseline
test_pred = best_pipeline.predict(X_te)
test_mae  = mean_absolute_error(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_r2   = r2_score(y_test, test_pred)

print("=" * 45)
print("  OFFICIAL WEEK 2 BASELINE — TEST SET")
print("=" * 45)
print(f"  Model     : {best_model_name}")
print(f"  Test MAE  : ${test_mae:>10,.0f}")
print(f"  Test RMSE : ${test_rmse:>10,.0f}")
print(f"  Test R²   : {test_r2:>10.4f}")
print("=" * 45)
print()
print("This number is the benchmark. Any future improvement")
print("(feature engineering, hyperparameter tuning) must")
print("beat this test RMSE to be considered a real gain.")

📝 **What's happening here**

**This is the most important cell in the notebook.** We touch the test set
exactly once, at the very end, after all model selection decisions have been made
using only the validation set.

**Why "once only" matters:**
- If we evaluated on the test set during model development (e.g., to pick
  the best model or the best feature count), we would be implicitly tuning to
  the test set.
- The resulting test RMSE would be *optimistically biased* — it would not
  reflect true performance on genuinely unseen data.
- The discipline of keeping the test set locked until the very end is what
  makes the final number trustworthy.

**The test set = the real world.** Every other evaluation in this notebook
(validation RMSE, CV RMSE) was an estimate. The test RMSE is the closest
approximation we have to "how well will this model perform next month on
a house it has never seen?" 

In [ ]:
# Predicted vs Actual scatter on test set
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: Predicted vs Actual
min_val = min(y_test.min(), test_pred.min())
max_val = max(y_test.max(), test_pred.max())
axes[0].scatter(y_test, test_pred, alpha=0.4, s=15, color="steelblue")
axes[0].plot([min_val, max_val], [min_val, max_val], "r--",
             linewidth=1.5, label="Perfect prediction")
axes[0].set_xlabel("Actual Sale Price ($)")
axes[0].set_ylabel("Predicted Sale Price ($)")
axes[0].set_title(f"Predicted vs Actual — Test Set ({best_model_name})")
axes[0].legend()

# Residual histogram
residuals_test = y_test - test_pred
axes[1].hist(residuals_test, bins=40, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red", linewidth=1.5, linestyle="--")
axes[1].axvline(residuals_test.mean(), color="orange", linewidth=1.5, linestyle="--",
                label=f"Mean: ${residuals_test.mean():+,.0f}")
axes[1].set_xlabel("Residual ($)")
axes[1].set_ylabel("Count")
axes[1].set_title("Test Set Residual Distribution")
axes[1].legend()

plt.suptitle(f"Test Set Evaluation — {best_model_name}", fontsize=13)
plt.tight_layout()
plt.show()

📝 **What's happening here**

**Left chart — Predicted vs Actual:**
- Points cluster along the 45° red line → correct predictions.
- Points above the line → model underestimated (predicted lower than actual).
- Points below the line → model overestimated.
- Systematic deviation at the high-price end is expected: very expensive houses
  ($400k+) are underrepresented in training data, so the model has seen fewer
  examples to learn from.

**Right chart — Test set residual distribution:**
- Should be approximately bell-shaped and centred near zero.
- The orange line shows the mean residual. A mean close to $0 means the model
  has no systematic directional bias (not consistently over/under-predicting
  across all price ranges).

## 11) Serialize Model

In [ ]:
# Save the best fitted pipeline
model_path = MODELS_DIR / "best_model_v1.joblib"
joblib.dump(best_pipeline, model_path)
print(f"Pipeline saved: {model_path}")
print(f"File size: {model_path.stat().st_size / 1024:.1f} KB")

# Save training statistics for the API layer
best_row = next(r for r in all_results if r["model"] == best_model_name)
training_stats = {
    "model_name":        best_model_name,
    "selected_features": selected_features,
    "test_mae":          float(test_mae),
    "test_rmse":         float(test_rmse),
    "test_r2":           float(test_r2),
    "val_mae":           float(best_row["val_mae"]),
    "val_rmse":          float(best_row["val_rmse"]),
    "val_r2":            float(best_row["val_r2"]),
    "n_train":           int(X_tr.shape[0]),
    "n_features_in":     int(X_tr.shape[1]),
}

stats_path = MODELS_DIR / "training_stats.json"
with open(stats_path, "w") as f:
    json.dump(training_stats, f, indent=2)
print(f"Training stats saved: {stats_path}")

📝 **What's happening here**

`joblib.dump()` serialises the entire fitted pipeline — the model with all its
learned parameters — to a single binary file. The file contains everything
needed to make predictions:
- The model type and all hyperparameters
- All fitted parameters (tree structures, coefficients, etc.)
- The list of expected input features (implicitly)

`training_stats.json` saves key metadata alongside the model. This is important
because the `.joblib` file alone does not tell you:
- What performance to expect
- Which features to pass at inference time
- When the model was trained

The stats file is what the FastAPI layer will read to populate the `/health`
endpoint and prediction response metadata.

In [ ]:
# Reload and smoke-test on 3 real validation rows
loaded_pipeline = joblib.load(model_path)

sample_X = X_v.iloc[:3]
sample_y = y_val.iloc[:3]
sample_preds = loaded_pipeline.predict(sample_X)

print("Smoke test — 3 validation rows through loaded pipeline:")
print(f"{'Row':<5} {'Actual':>12} {'Predicted':>12} {'Error':>12}")
print("-" * 45)
for i, (actual, pred) in enumerate(zip(sample_y, sample_preds)):
    err = pred - actual
    print(f"{i:<5} ${actual:>10,.0f} ${pred:>10,.0f} {err:>+11,.0f}")
print()
print(f"Loaded pipeline type: {type(loaded_pipeline).__name__}")
print(f"Model step          : {type(loaded_pipeline.named_steps['model']).__name__}")
print("Smoke test passed ✓")

📝 **What's happening here**

The smoke test verifies that the saved `.joblib` file is self-contained and
produces correct predictions. We:
1. Load the file from disk (simulating deployment on a new machine)
2. Pass 3 real validation rows through the loaded pipeline
3. Confirm predictions are reasonable and match what we expect

**Why test after loading (not just after saving)?**
`joblib.dump()` followed immediately by `joblib.load()` is the only way to
confirm the serialisation was successful. It is possible (rare but real) for a
file to be written but corrupted — the smoke test catches this immediately rather
than during production deployment.

## 12) Summary

### What we built in this notebook

| Step | What happened |
|------|---------------|
| §1 Load data | Loaded clean splits from `data/processed/` (output of NB02) |
| §2 Feature selection | Ranked features using Pearson r, F-test, and Mutual Information |
| §3 Pipelines | Built one sklearn Pipeline per model (data already preprocessed) |
| §4 eval_pipeline | Reusable helper: fit → predict → MAE/RMSE/R² on train + val |
| §5–7 Baselines | LinearRegression, RandomForest, GradientBoosting |
| §8 Overfitting table | Train vs val RMSE gap for all three models |
| §9 Comparison | Sorted by val RMSE; CV RMSE used for final selection |
| §10 Test evaluation | One-shot test evaluation → official Week 2 baseline |
| §11 Serialise | `models/best_model_v1.joblib` + `training_stats.json` |

### Key takeaways

- **Statistical feature selection before modeling** reduces noise and overfitting risk.
- **CV RMSE** is more reliable than a single val split for model selection.
- **The bias-variance tradeoff** is visible in the overfitting table: more model
  capacity → lower train RMSE, wider train-val gap, modestly better val RMSE.
- **The test set is evaluated exactly once.** That number is the official baseline.

### Next steps (Week 3)

- Hyperparameter tuning (GridSearchCV / RandomizedSearchCV)
- Log-transform the target to reduce heteroscedasticity
- Advanced feature engineering (interaction terms, polynomial features)
- Regularised linear models (Ridge, Lasso) to compare with tree-based methods
